In [ ]:
import numpy as np

arr = np.genfromtxt(
    "task_01_memory_clients.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8"
)


# Задача 1. Оптимизация памяти клиентской базы
Файл: `task_01_memory_clients.csv`

Поля:
- client_id
- age
- orders_total
- lifetime_revenue_rub
- days_since_last_order
- is_active
- segment

Задание:
1. Загрузите данные.
2. Для каждого поля предложите оптимальный NumPy dtype.
3. Посчитайте текущий объём памяти для числовых колонок.
4. Приведите массивы к более экономным типам.
5. Посчитайте экономию памяти в байтах и процентах.
6. Отдельно проверьте, не потерялся ли смысл данных после преобразования.


In [26]:
arr1 = np.genfromtxt(
    "task_01_memory_clients.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8"
)

# print(arr1.dtype)
# print(np.finfo(np.float16))

# print(arr1.dtype)
# print(arr1[:5])

# обращение к каждому столбцу строки
client_id = arr1["client_id"]
age = arr1["age"]
orders_total = arr1["orders_total"]
lifetime_revenue_rub = arr1["lifetime_revenue_rub"]
days_since_last_order = arr1["days_since_last_order"]
is_active = arr1["is_active"]
segment = arr1["segment"]

# Приводим типы к нужной размерности
client_id_opt = client_id.astype(np.uint32)
age_opt = age.astype(np.uint8)
orders_total_opt = orders_total.astype(np.uint16)
lifetime_revenue_rub_opt = lifetime_revenue_rub.astype(np.float32)
days_since_last_order_opt = days_since_last_order.astype(np.uint16)
is_active_opt = is_active.astype(np.bool_)
segment_opt = segment.astype(np.uint8)


# проверяем диапазоны
# print("client_id:", client_id.min(), client_id.max())
# print("age:", age.min(), age.max())
# print("orders_total:", orders_total.min(), orders_total.max())
# print("lifetime_revenue_rub:", lifetime_revenue_rub.min(), lifetime_revenue_rub.max())
# print("days_since_last_order:", days_since_last_order.min(), days_since_last_order.max())
# print("is_active:", is_active.min(), is_active.max())
# print("segment:", segment.min(), segment.max())


# считаем разницу в памяти
memory_before = (
    client_id.nbytes
    + age.nbytes
    + orders_total.nbytes
    + lifetime_revenue_rub.nbytes
    + days_since_last_order.nbytes
    + is_active.nbytes
    + segment.nbytes
)

memory_after = (
    client_id_opt.nbytes
    + age_opt.nbytes
    + orders_total_opt.nbytes
    + lifetime_revenue_rub_opt.nbytes
    + days_since_last_order_opt.nbytes
    + is_active_opt.nbytes
    + segment_opt.nbytes
)

print("Память до:", memory_before, "байт")
print("Память после:", memory_after, "байт")
print("Экономия:", memory_before - memory_after, "байт")
print("Экономия, %:", round((1 - memory_after / memory_before) * 100, 2))

# np.array_equal() проверяет, полностью ли одинаковые два массива:
print("client_id:", np.array_equal(client_id, client_id_opt))
print("age:", np.array_equal(age, age_opt))
print("orders_total:", np.array_equal(orders_total, orders_total_opt))
print("days_since_last_order:", np.array_equal(days_since_last_order, days_since_last_order_opt))
print("is_active:", np.array_equal(is_active.astype(bool), is_active_opt))
print("segment:", np.array_equal(segment, segment_opt))
print("lifetime_revenue_rub:", np.allclose(lifetime_revenue_rub, lifetime_revenue_rub_opt))


Память до: 2240 байт
Память после: 600 байт
Экономия: 1640 байт
Экономия, %: 73.21
client_id: True
age: True
orders_total: True
days_since_last_order: True
is_active: True
segment: True
lifetime_revenue_rub: True


# Задача 2. План-факт менеджеров
Файл: `task_02_sales_plan_fact.csv`

Поля:
- manager
- month
- sales_plan_rub
- sales_fact_rub
- new_clients_plan
- new_clients_fact
- op_kpi

Задание:
1. Получите массивы плана и факта продаж.
2. Посчитайте выполнение плана продаж.
3. Посчитайте выполнение плана по новым клиентам.
4. Рассчитайте итоговый индекс менеджера по формуле:
   - продажи - 50%
   - новые клиенты - 30%
   - op_kpi - 20%
5. Найдите строки, где итоговый индекс ниже 0.8.
6. Найдите строки, где продажи выполнены на 120% и выше.
7. Преобразуйте продажи в матрицу 8 менеджеров x 12 месяцев.
8. Посчитайте среднее выполнение по каждому менеджеру и по каждому месяцу.

In [62]:
arr2 = np.genfromtxt(
    "task_02_sales_plan_fact.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8"
)
# print(arr2[:3])


# no.column_stack - делает из структурированного массива матрицу (1+ мерную)

sales_plan_rub = arr2["sales_plan_rub"]
sales_fact_rub = arr2["sales_fact_rub"]

# считаем выполнение плана
#
sales_completion = sales_fact_rub / sales_plan_rub
# print(sales_completion)

# Выполнение плана по новым клиентам

new_sales_plan_rub = arr2["new_clients_plan"]
new_sales_fact_rub = arr2["new_clients_fact"]

new_sales_completion = new_sales_fact_rub / new_sales_plan_rub
# print(new_sales_completion)


# Итоговый индекс менеджера
op_kpi = arr2["op_kpi"]

manager_index = (
    sales_completion * 0.5
    + new_sales_completion * 0.3
    + op_kpi * 0.2
)

# print(manager_index)

# Найдите строки, где итоговый индекс ниже 0.8.
filtr = (manager_index < 0.8)

# print(filtr)
# print(manager_index[filtr])


# print(arr2["manager"][filtr])
# print(arr2["month"][filtr])
# print(manager_index[filtr])

# Найдите строки, где продажи выполнены на 120% и выше.

goods_sales = sales_completion >= 1.2

# print(sales_completion[goods_sales])

sales_fact_matrix = sales_completion.reshape(8, 12)

print(sales_fact_matrix)
# print(sales_fact_matrix.shape)

# Среднее выполнение по каждому менеджеру
# axis=1 - вправо по столбцам, то есть по каждой строке
avg_managers_sales_completion = sales_fact_matrix.mean(axis=1)
# print(avg_managers_sales_completion)

# axis=0 - вниз по строкам, то есть по каждому столбцу
mean_by_month = sales_fact_matrix.mean(axis=0)
# print(mean_by_month)

[[0.6532911  0.70272195 0.89749539 1.00584255 1.37026049 0.88525466
  1.33692854 1.15779163 0.7061923  1.06388163 1.38231892 1.36883478]
 [1.33334171 0.87274273 0.69557311 0.71820624 1.09950575 1.39746055
  0.80372478 1.17540957 1.19232507 0.93806248 1.38449776 1.12717966]
 [1.09683393 1.24781053 0.80611225 1.10248564 1.31399204 1.21681118
  0.73693848 1.35677613 0.63691811 1.09291469 1.32000352 0.80939587]
 [0.91889055 1.35987283 1.40365629 0.72764146 0.99023946 1.00230102
  0.91311323 1.3790859  1.04834747 1.12633327 1.17583795 1.20057171]
 [1.26070705 1.37682134 0.75703271 1.03810491 1.36498882 1.14629042
  1.33869318 0.87155178 0.92665577 1.24424068 1.38095522 0.71753734]
 [1.28031894 1.35475331 1.32335969 1.15254722 0.64319946 1.34851023
  0.6613261  0.66578501 1.17849639 1.02362434 0.74416993 1.40416367]
 [1.32731601 0.7668263  1.07793018 1.08241365 1.0266815  0.72727517
  1.12770312 1.09064387 0.80047379 1.13835824 0.96513406 1.20936374]
 [1.25989608 0.67764673 0.88730354 1.2817

# Задача 3. Маржинальность сделок
Файл: `task_03_orders_margin.csv`

Поля:
- order_id
- customer
- manager
- revenue_rub
- cost_rub
- days_to_ship

Задание:
1. Посчитайте маржу по каждой сделке.
2. Посчитайте маржинальность.
3. Найдите сделки с отрицательной маржой.
4. Найдите сделки с маржинальностью ниже 10%.
5. Найдите сделки с выручкой выше 1 млн руб. и маржинальностью ниже 15%.
6. Найдите сделку с максимальной маржей.
7. Найдите среднюю, медианную и минимальную маржинальность.
8. Проверьте, есть ли связь между коротким сроком отгрузки и низкой маржинальностью на уровне простой группировки по маске.

In [ ]:
arr3 = np.genfromtxt(
    "task_03_orders_margin.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8"
)

order = arr3[
    ["order_id", "customer", "manager", "revenue_rub", "cost_rub", "days_to_ship"]
]

order_rows = [list(row.tolist()) for row in order]

order_matrix = np.array(
    [list(row.tolist()) for row in order],
    dtype=object
)


# print(order_matrix)

# order_id,customer,manager,revenue_rub,cost_rub,days_to_ship

# print(arr3)

revenue_rub = arr3["revenue_rub"]
cost_rub = arr3["cost_rub"]

############# считаем маржинальность по каждой сделке #############
margin = revenue_rub - cost_rub
# print(margin)

order_matrix = np.column_stack([
    order_matrix,
    margin
])

# print(order_matrix)


############# считаем маржинальность #############
# sum_revenue_rub = sum(revenue_rub)
# sum_cost_rub = sum(cost_rub)
# sum_margin = sum_revenue_rub - sum_cost_rub

# print(sum_margin)


############# Найдите сделки с отрицательной маржой #############
# negative_mar = (margin < 0)
# negative_orders = order_matrix[negative_mar]
# print(negative_orders)


############# Найдите сделки с маржинальностью ниже 10% #############

margin_prec = margin / revenue_rub

filter_margin_prec = margin_prec < 0.1

order_matrix = np.column_stack([
    order_matrix,
    margin_prec
])

negative_margin_prec = order_matrix[filter_margin_prec]

# print(negative_margin_prec)


############# Найдите сделки с выручкой выше 1 млн руб. и маржинальностью ниже 15%. #############

filter_orders = (margin_prec < 0.15) & (revenue_rub > 1000000)
# print(order_matrix[filter_orders])


############# Найдите сделку с максимальной маржей. #############

sorted_by_margin = order_matrix[
    np.argsort(order_matrix[:, 6])[::-1]
]

# print(sorted_by_margin[1])


############# 7. Найдите среднюю, медианную и минимальную маржинальность. #############

avg_margin = order_matrix.mean()


TypeError: only integer scalar arrays can be converted to a scalar index